# Módulo 08 — Teoría Evolutiva de Juegos

**Objetivos**: Implementar la dinámica del replicador con `scipy.integrate.solve_ivp`. Analizar estabilidad de equilibrios y generar diagramas de fase para distintos ratios V/C en el juego Halcón-Paloma.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
print('Entorno listo.')

## 1. El juego Halcón-Paloma

Dos estrategias compiten por un recurso de valor V con coste de combate C:
- Halcón (H) vs Halcón: cada uno obtiene (V-C)/2
- Halcón vs Paloma: Halcón obtiene V, Paloma obtiene 0
- Paloma vs Paloma: cada uno obtiene V/2

La ESS depende del ratio V/C:
- Si V ≥ C: la ESS es 100% Halcones (estrategia pura dominante)
- Si V < C: la ESS es un polimorfismo mixto con fracción Halcones = V/C

In [ ]:
def hawk_dove_matrix(V, C):
    """Retorna la matriz de pagos del juego Halcón-Paloma."""
    return np.array([
        [(V - C) / 2,  V],    # Halcón vs (H, P)
        [0,            V / 2]  # Paloma vs (H, P)
    ])

def hawk_dove_ess(V, C):
    """Fracción de Halcones en la ESS. Retorna 1.0 si V >= C."""
    return 1.0 if V >= C else V / C

V, C = 6, 10
A = hawk_dove_matrix(V, C)
print(f'V={V}, C={C}  →  ESS Halcones = {hawk_dove_ess(V,C):.2f} ({hawk_dove_ess(V,C)*100:.1f}%)')
print('Matriz de pagos:')
print(A)

## 2. Dinámica del Replicador

La ecuación del replicador para n estrategias:

ẋᵢ = xᵢ · (fᵢ − f̄)

donde fᵢ = (Ax)ᵢ es el fitness de la estrategia i y f̄ = xᵀAx es el fitness medio de la población.

In [ ]:
def replicator_ode(t, x, A):
    """ODE del replicador. x = vector de frecuencias, A = matriz de pagos."""
    n = len(x)
    f = A @ x           # fitness de cada estrategia
    f_mean = x @ f      # fitness medio
    dxdt = x * (f - f_mean)
    return dxdt

def simulate_replicator(A, x0, T, dt=0.05):
    """Simula la dinámica del replicador desde x0 hasta t=T."""
    t_span = (0, T)
    t_eval = np.arange(0, T + dt, dt)
    # Normalizar x0
    x0 = np.array(x0, dtype=float)
    x0 = x0 / x0.sum()
    sol = solve_ivp(replicator_ode, t_span, x0, args=(A,), t_eval=t_eval, method='RK45')
    return sol.t, sol.y

# Simulación: 80% Palomas al inicio
x0 = [0.2, 0.8]  # [Halcones, Palomas]
t, y = simulate_replicator(A, x0, T=40)

ess = hawk_dove_ess(V, C)

plt.figure(figsize=(9, 4.5))
plt.plot(t, y[0], '#c0392b', lw=2.5, label='Halcones')
plt.plot(t, y[1], '#2d7a50', lw=2.5, label='Palomas')
plt.axhline(ess, color='#8b5cf6', ls='--', lw=1.5, label=f'ESS = {ess:.2f}')
plt.axhline(1-ess, color='#8b5cf6', ls=':', lw=1, alpha=0.5)
plt.xlabel('Generaciones'); plt.ylabel('Proporción en la población')
plt.title(f'Halcón-Paloma: V={V}, C={C}, x₀(Halcones)={x0[0]}')
plt.legend(); plt.grid(alpha=0.3); plt.ylim(0, 1)
plt.tight_layout(); plt.show()
print(f'Estado final: Halcones={y[0,-1]:.4f}, Palomas={y[1,-1]:.4f}  (ESS={ess:.4f})')

## 3. Diagrama de fases: múltiples condiciones iniciales

Muestra que desde cualquier punto de partida la población converge a la ESS.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, (V_i, C_i) in zip(axes, [(6, 10), (12, 8)]):
    A_i = hawk_dove_matrix(V_i, C_i)
    ess_i = hawk_dove_ess(V_i, C_i)

    for x0_h in np.linspace(0.05, 0.95, 8):
        x0 = [x0_h, 1-x0_h]
        t, y = simulate_replicator(A_i, x0, T=30)
        ax.plot(t, y[0], alpha=0.6, lw=1.5)

    ax.axhline(ess_i, color='#8b5cf6', ls='--', lw=2, label=f'ESS = {ess_i:.2f}')
    ax.set_xlabel('Generaciones'); ax.set_ylabel('Proporción Halcones')
    ax.set_title(f'V={V_i}, C={C_i}  ({"V>C: ESS=100% Halcones" if V_i>=C_i else f"ESS={ess_i:.2f}"})')
    ax.set_ylim(0, 1); ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Diagrama de fases: dinámica del replicador', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 4. Clasificar equilibrios: estabilidad

Un equilibrio x* es:
- **ESS**: estable ante perturbaciones pequeñas → atractor de la dinámica
- **Inestable**: cualquier perturbación aleja la población de x*
- **Neutro**: la dinámica es constante en un conjunto de equilibrios

In [ ]:
def classify_equilibria(A, n_points=200):
    """Encuentra y clasifica equilibrios del replicador para un juego 2-estrategias."""
    results = []
    for x0_h in np.linspace(0.001, 0.999, n_points):
        x0 = np.array([x0_h, 1-x0_h])
        t, y = simulate_replicator(A, x0, T=100, dt=0.1)
        x_final = y[0, -1]
        results.append((x0_h, x_final))
    return results

# Con V=6, C=10
res = classify_equilibria(hawk_dove_matrix(6, 10))
x0s, x_finals = zip(*res)
plt.figure(figsize=(7, 4))
plt.scatter(x0s, x_finals, s=10, alpha=0.5, color='#1a3a5c')
plt.axhline(hawk_dove_ess(6,10), color='#2d7a50', ls='--', lw=2, label=f'ESS = {hawk_dove_ess(6,10):.2f}')
plt.xlabel('Fracción inicial de Halcones (x₀)')
plt.ylabel('Fracción final de Halcones (t=100)')
plt.title('Convergencia a la ESS — Halcón-Paloma V=6, C=10')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Ejercicios

**Ejercicio 1**: Simula el juego Halcón-Paloma con V=15, C=10. ¿Cuál es la ESS? ¿Se extinguen las Palomas?

**Ejercicio 2**: El Dilema del Prisionero tiene la siguiente forma evolutiva: Cooperadores vs Desertores. A=[[3,0],[5,1]], x0=[0.5, 0.5]. Simula y explica el resultado.

In [ ]:
# Tu código aquí
